In [1]:
import numpy as np
import torch
import torchvision.transforms as T
# from decord import VideoReader, cpu
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = image_file.convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

model = AutoModel.from_pretrained(
    "5CD-AI/Vintern-1B-v2",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained("5CD-AI/Vintern-1B-v2", trust_remote_code=True, use_fast=False)

#question = "Câu hỏi khác ......"
#response, history = model.chat(tokenizer, pixel_values, question, generation_config, history=history, return_history=True)
#print(f'User: {question}\nAssistant: {response}')


/home/duckq1u/miniconda3/envs/OCR/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v2:
- configuration_internvl_chat.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v2:
- modeling_internvl_chat.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/home/duckq1u/miniconda3/envs/OCR/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, pleas

FlashAttention is not installed.


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [2]:

import os
from PIL import Image
from pdf2image import convert_from_path
from sacrebleu import corpus_bleu
from bert_score import score
import json

# NOTE: Change this to your folder path
folder_path = '/home/duckq1u/Documents/obsidian_aio/Notebook/Môn học trên trường/OCR thầy minh/data_test'
question = '''
###HỆ THỐNG:
- Bạn là hệ thống trích xuất thông tin pháp lý chính xác. 

###YÊU CẦU:
Trích xuất thông tin:
1. Số đăng ký
2. Bên nhận bảo đảm / Bên nhận thế chấp (phải có CCCD/CMND nếu là cá nhân)
3. Bên bảo đảm / Bên thế chấp (phải có CCCD/CMND nếu là cá nhân)
   
###ĐỊNH DẠNG KẾT QUẢ:
1. Số đăng ký: <thông tin>
2. Bên nhận bảo đảm / Bên nhận thế chấp: 
- <Tên tổ chức hoặc cá nhân>, Địa chỉ: <địa chỉ hoặc "Không xác định">, CCCD/CMND: <số hoặc "Không xác định">, Mã số thuế: <mã số thuế hoặc "Không xác định">
3. Bên bảo đảm / Bên thế chấp: 
- <Tên tổ chức hoặc cá nhân>, Địa chỉ: <địa chỉ hoặc "Không xác định">, CCCD/CMND: <số hoặc "Không xác định">, Mã số thuế: <mã số thuế hoặc "Không xác định">

* Lưu ý: Mỗi cá nhân/tổ chức bắt đầu dòng mới bằng dấu "-".
* Nếu không có thông tin, ghi "Không xác định".
* Không thêm, bớt, hay sửa đổi định dạng trên.

###QUY TẮC:
- Phải xác định đúng ai là tổ chức, ai là cá nhân (dựa vào địa chỉ, tên, hoặc dấu hiệu nhận biết).
- Mỗi cá nhân hoặc tổ chức phải có một dòng riêng bắt đầu bằng dấu `-`.
- BẮT BUỘC với cá nhân phải có CCCD/CMND, nếu không có phải ghi "Không xác định".
- Với tổ chức, có thể không có CCCD/CMND nhưng nếu có mã số thuế phải ghi đầy đủ, nếu không có ghi "Không xác định".
- Không được bỏ sót bất kỳ cá nhân hay tổ chức nào có trong văn bản.
- **Tuyệt đối không thêm thông tin ngoài định dạng trên, không thêm giải thích, chú thích.**
- Không thay đổi thứ tự các mục.
- Kết quả chỉ là văn bản thuần, không dùng Markdown hay HTML.
'''


# Open and load the file
with open(folder_path+'/infor.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    
print(type(data))

def get_text_from_json(data, item):
    result = ""
    for index, value in enumerate(data[f'{item}']): 
        key = list(data[f'{item}'].keys())[index]
        value = data[f'{item}'][key]
        result = f"{result}\n{key} {value}"
    return result

def evaluate_similarity(reference: str, candidate: str, lang="en"):
    # BLEU Score 
    bleu = corpus_bleu([candidate], [[reference]])
    print(f"[BLEU] Similarity Score: {bleu.score:.2f}")

    # BERTScore 
    P, R, F1 = score([candidate], [reference], lang=lang)
    print(f"[BERTScore] Precision: {P[0]:.4f}, Recall: {R[0]:.4f}, F1: {F1[0]:.4f}")

# Loop through all files in the folder
for index, filename in enumerate(os.listdir(folder_path)):
    if filename.endswith('.pdf'): # Ignore json files
        file_path = os.path.join(folder_path, filename) # Construct full file path
        print(file_path)
        # Convert PDF to images
        images = convert_from_path(file_path)
        image = images[0]
        result_paragraphs = get_text_from_json(data, int(filename.split('.')[0]))
        pixel_values = load_image(image, max_num=6).to(torch.bfloat16).cuda()
        generation_config = dict(max_new_tokens= 1024, do_sample=False, num_beams = 3, repetition_penalty=2.5)
        
        response, history = model.chat(tokenizer, pixel_values, question, generation_config, history=None, return_history=True)
        print(f'Assistant: {response}')
        evaluate_similarity(reference=result_paragraphs, candidate=response, lang="vi")
        print('*'*50)

<class 'dict'>
/home/duckq1u/Documents/obsidian_aio/Notebook/Môn học trên trường/OCR thầy minh/data_test/3.pdf


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


TypeError: Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151655, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen